In [1]:
from src.task_vectors import TaskVector
task_vector_A = TaskVector("/home/aelmusta/efficientnet-b3-5fb5a3c3.pth","/scratch/aelmusta/experiment-2-1.1-finetuning/lightning_logs/version_24020803/checkpoints/last.ckpt")

ModuleNotFoundError: No module named 'src'

In [2]:
# task_vector.vector

In [3]:
from src.task_vectors import TaskVector
task_vector_B=TaskVector("/home/aelmusta/efficientnet-b3-5fb5a3c3.pth","/home/aelmusta/active_learning/ftw-baselines/3_Class_FULL_FTW_Pretrained.ckpt")

False True


In [4]:
# task_vector_A.vector

In [5]:
new_task_vector = task_vector_A + task_vector_B

In [6]:
import segmentation_models_pytorch as smp
model = smp.Unet(
                encoder_name="efficientnet-b3",
                encoder_weights="imagenet",
                in_channels=8,
                classes=3,
            )

In [7]:
image_encoder = new_task_vector.apply_to("/home/aelmusta/efficientnet-b3-5fb5a3c3.pth", scaling_coef=0.8,model_cls=model)

/home/aelmusta/active_learning/ftw-baselines/task_vectors/src/task_vectors.py:108: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pretrained_model = torch.load(pretrained_che

In [12]:
import torch
torch.save(model.state_dict(), "/home/aelmusta/task_vectors.pth")

In [1]:
import torch
from ftw.trainers import CustomSemanticSegmentationTask

# Load the checkpoint dictionary
ckpt_path = "/home/aelmusta/task_vectors2.pth"
ckpt = torch.load(ckpt_path, map_location="cpu")

# Add a dummy pytorch-lightning version if it's missing
if "pytorch-lightning_version" not in ckpt:
    ckpt["pytorch-lightning_version"] = "1.7.0"  # you can choose a version string appropriate for your setup
torch.save(ckpt, "/home/aelmusta/task_vectors2.pth")
# Now load the model from the modified checkpoint dictionary
# model = CustomSemanticSegmentationTask.load_from_checkpoint(checkpoint_path=ckpt, map_location="cpu")

/tmp/ipykernel_73859/1376148970.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location="cpu")


In [ ]:
import torch
from task_vectors import TaskVector
from eval import eval_single_dataset
from args import parse_arguments

# Config
datasets = ['MNIST', 'RESISC45']
model = 'ViT-L-14'
args = parse_arguments()
args.data_location = '/path/to/data'
args.model = model
args.save = f'checkpoints/{model}'
pretrained_checkpoint = f'checkpoints/{model}/zeroshot.pt'

# Create the task vectors
task_vectors = [
    TaskVector(pretrained_checkpoint, f'checkpoints/{model}/{dataset}/finetuned.pt')
    for dataset in datasets
]
# Sum the task vectors
task_vector_sum = sum(task_vectors)
# Apply the resulting task vector
image_encoder = task_vector_sum.apply_to(pretrained_checkpoint, scaling_coef=0.8)
# Evaluate
for dataset in datasets:
    eval_single_dataset(image_encoder, dataset, args)